# Case Study §6 — Base vs Instruct, swept over SFT-set size (the centerpiece)

Runnable twin of [`06_base_vs_instruct_sweep.py`](06_base_vs_instruct_sweep.py).

**Claim under test** (unsloth guide's Recipe E): with a *small* SFT set, start from the **instruct**
model, not base — it already follows instructions, so a few examples suffice; base must learn the
answer format from scratch.

**Method:** hold out a fixed test set; for each init ∈ {base, instruct} and training size N, SFT
(completion-only loss) and measure **held-out completion perplexity** (lower = predicts gold answers
better). Same plain prompt-completion format for both, so only the *init* differs.

**Honesty:** we hand-authored 32 pairs, so N is capped (scarcity is the lesson). **TRIAL numbers are
noise** (few steps) — run `MODE="full"` for the real curve.

In [ ]:
MODE = "trial"     # "trial" or "full"
FORCE = False
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = next(p for p in [HERE, HERE/'scripts', HERE.parent/'scripts'] if (p/'config.py').exists())
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s6", root / "06_base_vs_instruct_sweep.py")
s6 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s6)
print(f"mode={config.RUN_MODE} sizes={config.limits()['sweep_sizes']}")

In [ ]:
m = s6.run(force=FORCE)

## Plot the curve (if matplotlib is available)
Perplexity vs N for base vs instruct. If the hypothesis holds, the instruct line sits below the base
line at small N and they converge as N grows.

In [ ]:
sizes = m['sizes']
base = [m['results']['base'][str(n)] for n in sizes]
inst = [m['results']['instruct'][str(n)] for n in sizes]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5,3))
    plt.plot(sizes, base, 'o-', label='base init')
    plt.plot(sizes, inst, 's-', label='instruct init')
    plt.xlabel('SFT training examples (N)'); plt.ylabel('held-out completion perplexity')
    plt.title(f'Base vs Instruct under small SFT data ({MODE})'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed - table form:')
    for n,b,i in zip(sizes, base, inst):
        print(f'  N={n:>3}  base={b:7.2f}  instruct={i:7.2f}  winner={"instruct" if i<b else "base"}')

## Verify

In [ ]:
assert m['results']['base'] and m['results']['instruct'], 'both inits must be swept'
assert all(v > 0 for v in base + inst), 'perplexities must be positive'
print('\u2713 §6 verified: base and instruct swept over', sizes, '; held-out perplexity recorded.')
print('Reminder: TRIAL is noise. The real base-vs-instruct verdict comes from MODE="full".')
print('Next: \u00a77 eval (perplexity + generations + rubric), then \u00a78 Unsloth-vs-HF, then Part B (edge).')